<a href="https://colab.research.google.com/github/Mridul-Anand-Geoscience/Predictive-Maintenance-Analysis/blob/main/Predictive_Maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. IMPORT THE TOOLKIT
# Pandas is like a super-powered Excel for Python
import pandas as pd

# 2. LOAD THE RAW DATA
# We are pulling real machinery sensor data directly from the internet
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)

# 3. CLEAN THE DATA
# We throw away columns that are just random ID numbers so they don't confuse our model
columns_to_drop = ['UDI', 'Product ID']
df_clean = df.drop(columns=columns_to_drop)

# 4. CHECK THE BREAKDOWN
# Let's see how many machines failed (1) vs operated normally (0)
print("Machine Status Breakdown:")
print(df_clean['Machine failure'].value_counts())

Machine Status Breakdown:
Machine failure
0    9661
1     339
Name: count, dtype: int64


Data Preprocessing and Train/Test Split
Machine learning models require numerical inputs. We encoded the categorical 'Type' column into integers. To prevent data leakage, we separated the target variable ('Machine failure') and dropped the individual failure mode flags. Finally, we split the dataset into an 80% training set and a 20% testing set to evaluate the model on unseen data.


In [2]:
# 1. TRANSLATE LETTERS TO NUMBERS
# We change L, M, and H into 0, 1, and 2 so the computer can do math with them
df_clean['Type'] = df_clean['Type'].map({'L': 0, 'M': 1, 'H': 2})

# 2. SEPARATE THE HINTS (X) FROM THE ANSWERS (y)
# We drop 'Machine failure' because that is the answer we want to guess.
# We also drop TWF, HDF, PWF, OSF, RNF because those are specific failure codes. If we leave them, the computer will cheat!
columns_to_hide = ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
X = df_clean.drop(columns=columns_to_hide)
y = df_clean['Machine failure']

# 3. SPLIT THE FLASHCARDS
# We import a tool that splits the data for us automatically
from sklearn.model_selection import train_test_split

# We keep 80% to train the computer, and hide 20% to test it later
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Flashcards for the computer to study: {X_train.shape[0]} machines")
print(f"Flashcards locked away for the final test: {X_test.shape[0]} machines")

Flashcards for the computer to study: 8000 machines
Flashcards locked away for the final test: 2000 machines


Model Training
We utilized a Random Forest Classifier to model the complex, non-linear relationships between the machine's sensor readings and catastrophic failure events. The model was trained exclusively on the 80% training subset.

In [3]:
# 1. IMPORT THE ALGORITHM
from sklearn.ensemble import RandomForestClassifier

# 2. CREATE THE MODEL
# We set up our Random Forest "council"
model = RandomForestClassifier(random_state=42)

# 3. TEACH THE MODEL
# We give it the 8000 flashcard hints (X_train) and the answers (y_train) to study
print("The computer is studying the sensor data... please wait...")
model.fit(X_train, y_train)
print("Studying complete! The model is trained.\n")

# 4. TAKE THE FINAL TEST
# Now we bring out the 2000 hidden flashcards (X_test) and ask it to guess the answers
predictions = model.predict(X_test)

# Let's peek at how it did on the very first 10 machines in the test!
print("Let's look at the first 10 test machines:")
print(f"Computer's Guesses: {predictions[:10]}")
print(f"Actual Answers:     {y_test[:10].values}")

The computer is studying the sensor data... please wait...
Studying complete! The model is trained.

Let's look at the first 10 test machines:
Computer's Guesses: [0 0 0 0 0 0 0 0 0 0]
Actual Answers:     [0 1 0 0 0 1 0 0 0 0]


Model Evaluation
To assess the model's performance on unseen data, we generated a classification report and a confusion matrix. Because failure events are rare, overall accuracy is a misleading metric. Instead, we must evaluate the model's ability to specifically catch the failure cases (Recall and Precision for class 1)

In [4]:
# 1. Import the scoring tools
from sklearn.metrics import classification_report, confusion_matrix

# 2. Print the Confusion Matrix (A grid showing right vs wrong guesses)
print("--- CONFUSION MATRIX ---")
print("(Top Left: Correctly guessed Normal | Top Right: False Alarms)")
print("(Bottom Left: Missed Failures     | Bottom Right: Correctly guessed Failure)\n")
print(confusion_matrix(y_test, predictions))

# 3. Print the Detailed Scores
print("\n--- DETAILED SCORE BREAKDOWN ---")
print(classification_report(y_test, predictions))

--- CONFUSION MATRIX ---
(Top Left: Correctly guessed Normal | Top Right: False Alarms)
(Bottom Left: Missed Failures     | Bottom Right: Correctly guessed Failure)

[[1933    6]
 [  25   36]]

--- DETAILED SCORE BREAKDOWN ---
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1939
           1       0.86      0.59      0.70        61

    accuracy                           0.98      2000
   macro avg       0.92      0.79      0.85      2000
weighted avg       0.98      0.98      0.98      2000



Handling Imbalanced Data
The initial model achieved 98% accuracy but suffered from low recall (59%) on actual failures due to severe class imbalance. To penalize the model for missing critical failures, we applied class weights, forcing the algorithm to prioritize the minority 'Failure' class.

In [5]:
# 1. RE-CREATE THE MODEL WITH "BALANCED" WEIGHTS
# This tells the computer that failures are extremely important
better_model = RandomForestClassifier(random_state=42, class_weight='balanced')

# 2. RE-TRAIN THE MODEL
better_model.fit(X_train, y_train)

# 3. TAKE THE TEST AGAIN
better_predictions = better_model.predict(X_test)

# 4. PRINT THE NEW SCORES
print("--- NEW BALANCED CONFUSION MATRIX ---")
print(confusion_matrix(y_test, better_predictions))

print("\n--- NEW DETAILED SCORE BREAKDOWN ---")
print(classification_report(y_test, better_predictions))

--- NEW BALANCED CONFUSION MATRIX ---
[[1935    4]
 [  32   29]]

--- NEW DETAILED SCORE BREAKDOWN ---
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1939
           1       0.88      0.48      0.62        61

    accuracy                           0.98      2000
   macro avg       0.93      0.74      0.80      2000
weighted avg       0.98      0.98      0.98      2000



SMOTE (Synthetic Minority Over-sampling Technique)
Because class weights were insufficient for the severe imbalance, we applied SMOTE exclusively to the training data. This generated synthetic minority class samples (failures), ensuring the Random Forest model trained on a perfectly balanced 50/50 distribution without leaking test data.

In [6]:
# 1. IMPORT SMOTE
from imblearn.over_sampling import SMOTE

# 2. CREATE FAKE FAILURES (ONLY IN THE TRAINING SET)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Old training size: {X_train.shape[0]} machines")
print(f"New SMOTE training size: {X_train_smote.shape[0]} machines (Perfect 50/50 Balance)\n")

# 3. TRAIN A BRAND NEW MODEL ON THE SMOTE DATA
final_model = RandomForestClassifier(random_state=42)
final_model.fit(X_train_smote, y_train_smote)

# 4. TAKE THE TEST
final_predictions = final_model.predict(X_test)

# 5. PRINT THE FINAL SCORES
print("--- FINAL SMOTE CONFUSION MATRIX ---")
print(confusion_matrix(y_test, final_predictions))

print("\n--- FINAL SMOTE DETAILED SCORE BREAKDOWN ---")
print(classification_report(y_test, final_predictions))

Old training size: 8000 machines
New SMOTE training size: 15444 machines (Perfect 50/50 Balance)

--- FINAL SMOTE CONFUSION MATRIX ---
[[1875   64]
 [  21   40]]

--- FINAL SMOTE DETAILED SCORE BREAKDOWN ---
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      1939
           1       0.38      0.66      0.48        61

    accuracy                           0.96      2000
   macro avg       0.69      0.81      0.73      2000
weighted avg       0.97      0.96      0.96      2000



Interactive Business Intelligence Visualization (Plotly)
While the numbers confirm model accuracy, we utilize Plotly to generate interactive charts for deeper stakeholder engagement. The first chart, **Feature Importance**, provides crucial business intelligence by identifying exactly which sensor readings (e.g., Torque, Rotational Speed) are the primary drivers of machinery failure. This is critical for preventative maintenance strategies.

In [7]:
import plotly.express as px

# 1. Get the predictive power scores from our final final model
importances = final_model.feature_importances_

# 2. Organize the scores into a neat table
feature_importance_df = pd.DataFrame({
    'Sensor/Feature': X.columns,
    'Predictive Power': importances
}).sort_values(by='Predictive Power', ascending=True) # Sort Ascending for horizontal bar

# 3. Create the interactive bar chart
fig_importance = px.bar(feature_importance_df,
                         x='Predictive Power',
                         y='Sensor/Feature',
                         orientation='h',
                         title='Business Intelligence: Which Sensors Drive Failure?',
                         labels={'Predictive Power': 'Model Predictive Power', 'Sensor/Feature': 'Sensor Reading'},
                         color='Predictive Power', # Add a color gradient
                         color_continuous_scale='Turbo', # Choose a colorful scale
                         template='plotly_white') # Clean white background

# Improve the layout and text display
fig_importance.update_layout(title_font_size=24, yaxis_tickfont_size=16)
fig_importance.update_traces(texttemplate='%{x:.2f}', textposition='outside') # Show scores on bars

# 4. Display the interactive chart
fig_importance.show()

Interactive Risk Mapping
To make the model's findings actionable for operations teams, we mapped the top predictive features (Torque vs. Rotational Speed). Using Plotly, this scatter map allows stakeholders to isolate failure events by toggling the legend and hovering over individual machines to see their precise sensor readouts prior to failure.

In [8]:
# 1. Translate the 0s and 1s into words so the chart looks professional
df_clean['Status'] = df_clean['Machine failure'].map({0: 'Normal Operation', 1: 'CATASTROPHIC FAILURE'})

# 2. Build the Interactive Plot
fig_scatter = px.scatter(
    df_clean,
    x='Rotational speed [rpm]',
    y='Torque [Nm]',
    color='Status', # This automatically creates our interactive legend!
    color_discrete_map={'Normal Operation': '#a6c8ff', 'CATASTROPHIC FAILURE': '#ff0000'}, # Light blue vs Bright Red
    hover_data=['Tool wear [min]', 'Air temperature [K]'], # Extra info when you hover
    title='Interactive Machine Risk Map (Click the Legend on the Right!)',
    template='plotly_white'
)

# Make the dots a bit larger so they are easy to hover over
fig_scatter.update_traces(marker=dict(size=8, opacity=0.8))

# 3. Display the chart
fig_scatter.show()